# Clasificador Naive Bayes - Lucas Nahuel Llanes

**Año:** 2026  
**Materia:** Aprendizaje Automático  
**Docente:** Scudero, Yanina Ximena  
**Carrera:** Tecnicatura en Ciencia de Datos e Inteligencia Artificial  
**Institución:** INSTITUTO SUPERIOR DE FORMACIÓN TÉCNICA N° 197

---

En este notebook se desarrolla un clasificador **Naive Bayes Multinomial** para distinguir noticias **reales** de **falsas**. El flujo incluye carga de datos, vectorización, entrenamiento, evaluación con split y validación cruzada, y predicción sobre noticias nuevas.

## 1. Carga y exploración del dataset

Se construye un `DataFrame` de pandas con 10 noticias etiquetadas. La columna `texto` contiene la noticia y `categoria` indica si es `real` o `fake`.

In [ ]:
import pandas as pd

noticias = {
    'texto': [
        "La NASA confirma el hallazgo de agua en Marte",
        "Descubren que la vacuna convierte a las personas en robots",
        "El ministerio de salud lanza campaña contra el dengue",
        "Científicos afirman que la Tierra es plana",
        "Se inaugura el nuevo hospital en la ciudad",
        "Celebridades usan crema milagrosa para rejuvenecer 30 años",
        "Gobierno aprueba ley de protección ambiental",
        "Estudio revela que comer chocolate cura el cáncer",
        "El presidente anunció una nueva reforma educativa",
        "Investigadores aseguran que los teléfonos espían nuestros sueños"
    ],
    'categoria': ["real", "fake", "real", "fake", "real",
                  "fake", "real", "fake", "real", "fake"]
}

df = pd.DataFrame(noticias)
X = df['texto']
y = df['categoria']
df

## 2. División en entrenamiento y prueba

Se usa `train_test_split` con `test_size=0.33` (33% prueba, 67% entrenamiento) y `random_state=2026` para que la división sea reproducible.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=2026
)
print("Entrenamiento:\n", X_train, "\n", y_train)
print("\nPrueba:\n", X_test, "\n", y_test)

Con `test_size=0.33` sobre 10 registros, quedan **6 para entrenamiento** y **4 para prueba**.

## 3. Vectorización y entrenamiento del modelo

Naive Bayes tiene tres variantes principales:

- **MultinomialNB** → conteo de palabras (texto).
- **GaussianNB** → variables numéricas continuas.
- **CategoricalNB** → variables categóricas discretas.

Como trabajamos con texto, usamos **MultinomialNB** junto con `CountVectorizer` para convertir las palabras en una matriz numérica de conteos.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vectorizador = CountVectorizer()
X_train_vec = vectorizador.fit_transform(X_train)

modelo = MultinomialNB()
modelo.fit(X_train_vec, y_train)

## 4. Evaluación del modelo

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

X_test_vec = vectorizador.transform(X_test)
y_pred = modelo.predict(X_test_vec)

print("Reporte de clasificación:\n", classification_report(y_test, y_pred))
print("Exactitud:", accuracy_score(y_test, y_pred))

### Interpretación de métricas

- **Precision:** de las noticias que el modelo marcó como `fake`, ¿cuántas lo eran realmente?
- **Recall:** de todas las noticias `fake` reales, ¿cuántas detectó?
- **F1-score:** media armónica entre precision y recall.
- **Accuracy:** proporción total de aciertos.

Con 4 noticias de prueba (2 reales y 2 falsas), el modelo obtuvo una exactitud de **1.0**, clasificando correctamente todas las instancias.

### Matriz de confusión

Permite ver gráficamente cuántas predicciones fueron correctas por clase. Se usa `confusion_matrix` de scikit-learn y `seaborn` para el heatmap.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred, labels=modelo.classes_)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=modelo.classes_,
            yticklabels=modelo.classes_)
plt.xlabel('Predicción')
plt.ylabel('Valor real')
plt.title('Matriz de confusión - Naive Bayes')
plt.tight_layout()
plt.show()

El modelo clasificó correctamente el 100% de las noticias del conjunto de prueba: la diagonal principal concentra los 4 aciertos y no hay errores fuera de ella.

## 5. Validación cruzada

Para una evaluación más robusta se aplica **StratifiedKFold** con 5 particiones sobre el dataset completo, dentro de un `Pipeline` que integra vectorizador y clasificador.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

pipeline = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('classifier', MultinomialNB())
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')

print("Precisión por fold:", scores)
print("Precisión media   :", scores.mean())
print("Desviación estándar:", scores.std())

## 6. Predicción sobre noticias nuevas

Se utiliza el modelo ya entrenado para clasificar tres noticias inéditas.

In [ ]:
nuevas_noticias = [
    "Nuevo estudio demuestra que el café mejora la memoria",
    "Expertos afirman que los gatos pueden hablar con humanos",
    "El congreso aprueba una nueva ley de educación"
]

nuevas_vec = vectorizador.transform(nuevas_noticias)
predicciones = modelo.predict(nuevas_vec)

for noticia, etiqueta in zip(nuevas_noticias, predicciones):
    print(f"- {noticia}  ->  {etiqueta}")

## 7. Conclusiones

El clasificador Naive Bayes Multinomial logró una exactitud perfecta sobre el conjunto de prueba, aunque el dataset es muy reducido. La validación cruzada mostró una precisión media cercana al 80% con variabilidad entre folds, lo que refleja la sensibilidad del modelo ante pocos datos. Como trabajo futuro se sugiere ampliar el corpus, aplicar TF-IDF y evaluar con métricas por clase.